<a href="https://colab.research.google.com/github/AileenLavelle/PBC_Object_Detection/blob/main/Replicating_Roboflow_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Jupiter Inlet Boat Detection - YOLO11

This notebook replicates Roboflow's training configuration to achieve:
- mAP@50: 83.8%
- Precision: 86.0%
- Recall: 76.0%

**Critical settings:**
- `imgsz=2048` (must match Roboflow's resize)
- `rect=True` (for non-square 2048x800 images)

## Step 1: Install Dependencies

In [8]:
!pip install -q opencv-python-headless
!pip install -q ultralytics
!pip install -q sahi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 3.3 MB/s eta 0:00:00


## Step 2: Import Libraries

In [9]:
import os
import json
import shutil
import random
from pathlib import Path
from copy import deepcopy

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
from ultralytics import YOLO

## Step 6: Configure Paths

In [10]:
# Auto-detect JSON file
json_files = [f for f in os.listdir('.') if f.endswith('.json') and 'annotation' in f.lower()]

if json_files:
    COCO_JSON = f"./{json_files[0]}"
else:
    COCO_JSON = "/content/_annotations.coco.json"  # Default name

IMAGES_DIR = "/content/Jupiter_Inlet"
OUTPUT_DIR = "/content/Jupiter_Inlet/jupiter_inlet_yolo"

print(f"COCO JSON: {COCO_JSON}")
print(f"Images dir: {IMAGES_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

COCO JSON: ./_annotations.coco.json
Images dir: /content/Jupiter_Inlet
Output dir: /content/Jupiter_Inlet/jupiter_inlet_yolo


## Step 7: Load COCO Annotations (with error handling)

In [11]:
def load_coco_json(json_path):
    """Safely load COCO JSON with encoding handling"""
    print(f"Loading: {json_path}")

    # Try different encodings
    for encoding in ['utf-8-sig', 'utf-8', 'latin-1']:
        try:
            with open(json_path, 'r', encoding=encoding) as f:
                content = f.read().strip()

            # Remove BOM if present
            if content.startswith('\ufeff'):
                content = content[1:]

            data = json.loads(content)
            print(f"Successfully loaded with {encoding} encoding")
            return data
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            print(f"Failed with {encoding}: {e}")
            continue

    raise ValueError("Could not parse JSON file")

# Load the annotations
coco_data = load_coco_json(COCO_JSON)

# Display info
print(f"\n{'='*50}")
print(f"Dataset Summary:")
print(f"{'='*50}")
print(f"  Images: {len(coco_data['images'])}")
print(f"  Annotations: {len(coco_data['annotations'])}")
print(f"  Categories: {coco_data['categories']}")

if coco_data['images']:
    img = coco_data['images'][0]
    print(f"  Image size: {img['width']} x {img['height']}")

Loading: ./_annotations.coco.json
Successfully loaded with utf-8-sig encoding

Dataset Summary:
  Images: 366
  Annotations: 1199
  Categories: [{'id': 0, 'name': 'Boat', 'supercategory': 'none'}, {'id': 1, 'name': 'Boat', 'supercategory': 'Boat'}]
  Image size: 4352 x 3264


## Step 8: Convert COCO to YOLO Format

In [13]:
def find_images_directory():
    """Find where images are located"""
    search_paths = ['./images', '.', './train', './valid', './test']

    for path in search_paths:
        if os.path.exists(path):
            for root, dirs, files_list in os.walk(path):
                jpg_files = [f for f in files_list if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
                if jpg_files:
                    print(f"Found {len(jpg_files)} images in: {root}")
                    return root
    return './images'


def coco_to_yolo(coco_data, images_dir, output_dir):
    """Convert COCO format to YOLO format"""

    output_path = Path(output_dir)

    # Create directories
    for split in ['train', 'val']:
        (output_path / 'images' / split).mkdir(parents=True, exist_ok=True)
        (output_path / 'labels' / split).mkdir(parents=True, exist_ok=True)

    # Build mappings
    image_map = {img['id']: img for img in coco_data['images']}

    annotations_by_image = {}
    for ann in coco_data['annotations']:
        img_id = ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id] = []
        annotations_by_image[img_id].append(ann)

    # 80/20 split
    image_ids = list(image_map.keys())
    split_idx = int(len(image_ids) * 0.8)
    train_ids = set(image_ids[:split_idx])

    print(f"Train: {len(train_ids)}, Val: {len(image_ids) - len(train_ids)}")

    # Find images
    images_path = Path(images_dir)
    all_image_files = {}
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        for f in images_path.rglob(ext):
            all_image_files[f.name] = f

    print(f"Found {len(all_image_files)} image files")

    processed = 0
    missing = 0

    for img_id, img_info in image_map.items():
        img_width = img_info['width']
        img_height = img_info['height']
        img_filename = img_info['file_name']

        split = 'train' if img_id in train_ids else 'val'

        # Find the image
        src_img = None
        basename = Path(img_filename).name

        if basename in all_image_files:
            src_img = all_image_files[basename]
        elif img_filename in all_image_files:
            src_img = all_image_files[img_filename]

        if src_img is None:
            missing += 1
            if missing <= 3:
                print(f"  Missing: {img_filename}")
            continue

        # Copy image
        dst_img = output_path / 'images' / split / basename
        shutil.copy2(src_img, dst_img)

        # Convert annotations to YOLO format
        label_filename = Path(basename).stem + '.txt'
        label_path = output_path / 'labels' / split / label_filename

        yolo_lines = []
        if img_id in annotations_by_image:
            for ann in annotations_by_image[img_id]:
                x_min, y_min, bbox_w, bbox_h = ann['bbox']

                # YOLO format: class x_center y_center width height (all normalized)
                x_center = (x_min + bbox_w / 2) / img_width
                y_center = (y_min + bbox_h / 2) / img_height
                w_norm = bbox_w / img_width
                h_norm = bbox_h / img_height

                # Clamp to [0, 1]
                x_center = max(0, min(1, x_center))
                y_center = max(0, min(1, y_center))
                w_norm = max(0, min(1, w_norm))
                h_norm = max(0, min(1, h_norm))

                # Class 0 (YOLO is 0-indexed)
                yolo_lines.append(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

        with open(label_path, 'w') as f:
            f.write('\n'.join(yolo_lines))

        processed += 1

    print(f"\nProcessed: {processed}, Missing: {missing}")

    # Create data.yaml
    data_yaml = f"""path: {output_path.absolute()}
train: images/train
val: images/val
nc: 1
names:
  0: Boat
"""

    yaml_path = output_path / 'data.yaml'
    with open(yaml_path, 'w') as f:
        f.write(data_yaml)

    print(f"Created: {yaml_path}")
    return str(yaml_path)


# Find images and convert
IMAGES_DIR = find_images_directory()
data_yaml_path = coco_to_yolo(coco_data, IMAGES_DIR, OUTPUT_DIR)

Found 74 images in: ./Jupiter_Inletjupiter_inlet_yolo/images/val
Train: 292, Val: 74
Found 74 image files
  Missing: l160855h_jpg.rf.29b340fb58c089bdd5009b9516d0806d.jpg
  Missing: s251110a_jpg.rf.035d1386ce4a1b41d5563de1683c5cbc.jpg
  Missing: o101205w_jpg.rf.35f41154b062b771959444c36742f602.jpg

Processed: 74, Missing: 292
Created: /content/Jupiter_Inlet/jupiter_inlet_yolo/data.yaml


## Step 9: Verify Conversion

In [14]:
print("=" * 50)
print("Verification")
print("=" * 50)

train_images = list(Path(OUTPUT_DIR).glob('images/train/*'))
val_images = list(Path(OUTPUT_DIR).glob('images/val/*'))
train_labels = list(Path(OUTPUT_DIR).glob('labels/train/*'))
val_labels = list(Path(OUTPUT_DIR).glob('labels/val/*'))

print(f"Train images: {len(train_images)}")
print(f"Val images:   {len(val_images)}")
print(f"Train labels: {len(train_labels)}")
print(f"Val labels:   {len(val_labels)}")

# Show sample label
if train_labels:
    sample = train_labels[0]
    print(f"\nSample label ({sample.name}):")
    with open(sample) as f:
        content = f.read()
        print(content[:500] if len(content) > 500 else content)

Verification
Train images: 0
Val images:   74
Train labels: 0
Val labels:   74


## Step 10: Train YOLO11 Model

**CRITICAL SETTINGS:**
- `imgsz=2048` - Must match Roboflow's resize
- `rect=True` - For non-square images (2048x800)

In [15]:
# Load pretrained YOLO11s
model = YOLO('yolo11s.pt')

# Train with Roboflow-matching settings
results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=2048,           # CRITICAL: Match Roboflow
    batch=4,              # Reduce if OOM (try 2 or 1)
    rect=True,            # CRITICAL: For non-square images
    device=0,             # GPU

    # Small object optimizations
    mosaic=0.5,
    scale=0.2,

    # Augmentations
    fliplr=0.5,
    flipud=0.0,

    # Training
    patience=50,
    save=True,
    plots=True,

    # Project
    project='jupiter_inlet',
    name='train',
)

Ultralytics 8.4.11 🚀 Python-3.12.12 torch-2.9.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: 0
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


## Step 11: Validate and Compare to Roboflow

In [ ]:
# Load best weights
best_weights = 'jupiter_inlet/train/weights/best.pt'
model = YOLO(best_weights)

# Validate
metrics = model.val(
    data=data_yaml_path,
    imgsz=2048,
    rect=True,
    conf=0.001,
    iou=0.5,
)

print("\n" + "=" * 50)
print("YOUR RESULTS:")
print(f"  mAP@50:    {metrics.box.map50:.1%}")
print(f"  Precision: {metrics.box.mp:.1%}")
print(f"  Recall:    {metrics.box.mr:.1%}")
print("=" * 50)
print("ROBOFLOW REPORTED:")
print("  mAP@50:    83.8%")
print("  Precision: 86.0%")
print("  Recall:    76.0%")
print("=" * 50)

## Step 12: Test Inference

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Get a test image
test_images = list(Path(OUTPUT_DIR).glob('images/val/*.jpg'))

if test_images:
    test_img = str(test_images[0])
    print(f"Testing on: {test_img}")

    # Run inference
    results = model.predict(
        source=test_img,
        imgsz=2048,
        conf=0.25,
        save=True,
    )

    print(f"Detected {len(results[0].boxes)} boats")

    # Display
    result_img = results[0].plot()
    plt.figure(figsize=(15, 6))
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f'Detected {len(results[0].boxes)} boats')
    plt.show()
else:
    print("No test images found")

## Step 13: Validate Your Existing Weights (Optional)

If you uploaded your `weights.pt` from Roboflow, run this cell to validate them:

In [ ]:
# Check if weights.pt exists
weights_files = [f for f in os.listdir('.') if f.endswith('.pt') and 'weight' in f.lower()]

if weights_files:
    weights_path = weights_files[0]
    print(f"Found weights: {weights_path}")

    # Load and validate
    model = YOLO(weights_path)

    print(f"\nModel info:")
    print(f"  Classes: {model.names}")

    # Validate with CORRECT settings
    metrics = model.val(
        data=data_yaml_path,
        imgsz=2048,      # CRITICAL!
        rect=True,       # CRITICAL!
        conf=0.001,
        iou=0.5,
    )

    print("\n" + "=" * 50)
    print("VALIDATION WITH YOUR WEIGHTS:")
    print(f"  mAP@50:    {metrics.box.map50:.1%}")
    print(f"  Precision: {metrics.box.mp:.1%}")
    print(f"  Recall:    {metrics.box.mr:.1%}")
    print("=" * 50)
else:
    print("No weights.pt file found. Upload one to validate.")

## Step 14: Download Trained Model

In [ ]:
if IN_COLAB:
    weights_path = 'jupiter_inlet/train/weights/best.pt'
    if os.path.exists(weights_path):
        files.download(weights_path)
        print("Download started!")
    else:
        print(f"Weights not found at {weights_path}")
        print("Available files:")
        !find . -name "*.pt" -type f

---

## Troubleshooting

### If metrics don't match Roboflow:

1. **Check image size**: Must be `imgsz=2048`
2. **Check rect mode**: Must be `rect=True`
3. **Check class mapping**: YOLO uses 0-indexed classes
4. **Check val split**: Roboflow may have different train/val split

### If you get OOM (Out of Memory):

```python
# Reduce batch size
model.train(..., batch=2)  # or batch=1
```

### If images aren't found:

```python
# List all files to find where images are
!find . -name "*.jpg" | head -20
```